# Análisis exploratorio de los datos

## Objetivo

Este notebook desarrolla el **análisis exploratorio de datos (EDA)** sobre el conjunto preprocesado.

Su finalidad consiste en describir la distribución del precio y examinar su vínculo con las principales características de las viviendas antes de estimar los modelos de regresión. Para mantener la coherencia con la modelización posterior, la comparación se limita a **pisos (`propertyType = "flat"`)**.

Las relaciones identificadas en esta etapa son **descriptivas**. Un gráfico o una correlación no bastan para establecer causalidad ni sustituyen el análisis multivariante del Notebook 04.

## 1. Importación de librerías

Se utilizan `pandas` y `numpy` para el análisis, y `matplotlib`/`seaborn` para las representaciones gráficas.

In [ ]:
# =====================================================
# IMPORTACIÓN DE LIBRERÍAS
# =====================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.max_columns", 100)

### Interpretación

Las librerías de análisis y visualización se han cargado correctamente. `pandas` y `numpy` se emplearán para preparar los datos, y `matplotlib` y `seaborn` para representar gráficamente las principales relaciones observadas.

## 2. Carga del conjunto de datos

Se carga `viviendas_limpio.csv`, generado en el Notebook 02. Este archivo reúne todas las tipologías inmobiliarias y conserva los valores ausentes relevantes sin una imputación general.

In [ ]:
# =====================================================
# CARGA DEL CONJUNTO DE DATOS
# =====================================================

# =====================================================
# DETECCIÓN DE LA RAÍZ DEL PROYECTO
# =====================================================

# Estructura esperada:
# TFM-Analisis-Inmobiliario/
# ├── data/raw
# ├── data/processed
# ├── notebooks
# └── reports/{figures,tables}
#
# La detección funciona tanto si VS Code ejecuta el notebook
# desde la raíz del proyecto como desde la carpeta notebooks/.
cwd = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        ruta for ruta in [cwd, *cwd.parents]
        if (ruta / "data" / "raw").is_dir()
        and (ruta / "notebooks").is_dir()
        and (ruta / "reports").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "No se ha podido localizar la raíz del proyecto. "
        "Se esperaba encontrar las carpetas data/raw, notebooks y reports "
        "en el directorio actual o en alguno de sus directorios superiores."
    )

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "viviendas_limpio.csv"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "No se encuentra viviendas_limpio.csv. "
        f"Ejecuta primero el Notebook 02. Ruta esperada: {DATA_PATH}"
    )

print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Archivo procesado: {DATA_PATH}")
print(f"Figuras: {FIGURES_DIR}")

df_viviendas = pd.read_csv(DATA_PATH, low_memory=False)

print(f"Registros cargados: {len(df_viviendas):,}")
print(f"Variables: {df_viviendas.shape[1]}")
display(df_viviendas.head())


### Interpretación

El conjunto preprocesado contiene **54.004 anuncios y 62 variables**. A partir de este dataset se define la muestra analítica de pisos empleada en el EDA.

## 3. Definición de la muestra de análisis

Solo se seleccionan los anuncios clasificados como `flat`. Esta decisión permite trabajar con una tipología homogénea y evita mezclar pisos con otras categorías del dataset, como chalets, áticos, dúplex, estudios o casas de campo.

Se distinguen dos muestras:

- **Muestra EDA inicial:** todos los pisos, utilizada para observar la distribución del precio antes del tratamiento de valores extremos.
- **Muestra comparativa:** pisos con variables clave completas y situados entre los percentiles 1 y 99 de `price` y `size` dentro de cada ciudad. Esta segunda muestra coincide con el criterio aplicado después en la modelización.

In [ ]:
# =====================================================
# MUESTRA DE PISOS
# =====================================================

df_eda = df_viviendas[
    df_viviendas["propertyType"].astype(str).str.lower().eq("flat")
].copy()

print(f"Pisos disponibles: {len(df_eda):,}")

variables_clave = [
    "price", "size", "rooms", "bathrooms",
    "ciudad", "latitude", "longitude",
]

df_base = (
    df_eda
    .dropna(subset=variables_clave)
    .loc[lambda x: (x["price"] > 0) & (x["size"] > 0)]
    .copy()
)

partes = []
resumen_muestra = []

for ciudad, grupo in df_base.groupby("ciudad", sort=True):
    p_price = grupo["price"].quantile([0.01, 0.99])
    p_size = grupo["size"].quantile([0.01, 0.99])

    mascara = (
        grupo["price"].between(p_price.loc[0.01], p_price.loc[0.99])
        & grupo["size"].between(p_size.loc[0.01], p_size.loc[0.99])
    )

    grupo_filtrado = grupo.loc[mascara].copy()
    partes.append(grupo_filtrado)

    resumen_muestra.append({
        "Ciudad": ciudad,
        "N inicial": len(grupo),
        "N comparativa": len(grupo_filtrado),
        "Retención (%)": 100 * len(grupo_filtrado) / len(grupo),
    })

df_analisis = pd.concat(partes, ignore_index=True)
resumen_muestra = pd.DataFrame(resumen_muestra)

display(resumen_muestra.round(2))
print(f"Total muestra comparativa: {len(df_analisis):,}")
# Figura 3-3: muestra por ciudad antes y después del filtrado
plot_muestra = (
    resumen_muestra
    .set_index("Ciudad")[["N inicial", "N comparativa"]]
)

ax = plot_muestra.plot(
    kind="bar",
    figsize=(12, 6),
)
ax.set_title("Tamaño de la muestra por ciudad antes y después del filtrado")
ax.set_xlabel("Ciudad")
ax.set_ylabel("Número de pisos")
plt.xticks(rotation=45, ha="right")

for i, fila in resumen_muestra.reset_index(drop=True).iterrows():
    altura = max(fila["N inicial"], fila["N comparativa"])
    ax.text(
        i,
        altura * 1.01,
        f'{fila["Retención (%)"]:.1f}%',
        ha="center",
        va="bottom",
        fontsize=8,
    )

plt.tight_layout()
ax.figure.savefig(
    FIGURES_DIR / "figura_3_03_muestra_por_ciudad_outliers.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


### Interpretación

De los **54.004 anuncios** del conjunto procesado, **43.891 son pisos**. Tras exigir valores completos en las variables clave y aplicar el mismo criterio de valores extremos que se utilizará en la modelización, la muestra comparativa queda formada por **42.441 observaciones**.

La retención se sitúa aproximadamente entre el **95 % y el 97 %** en todas las ciudades. **Ávila** registra el menor número de pisos, por lo que sus resultados deben interpretarse con mayor cautela.

## 4. Análisis de la variable objetivo: precio de oferta

### 4.1 Estadísticas descriptivas

Antes de aplicar filtros para visualización se describe el precio de los 43.891 pisos.

In [ ]:
# =====================================================
# ESTADÍSTICAS DESCRIPTIVAS DEL PRECIO
# =====================================================

estadisticas_precio = (
    df_eda["price"]
    .describe(percentiles=[0.90, 0.95, 0.99])
    .to_frame("precio de oferta")
    .round(2)
)

estadisticas_precio

### Interpretación

El precio muestra una dispersión elevada. La **mediana es 379.000 €**, mientras que la media se acerca a **600.000 €**. La diferencia entre ambas medidas y la presencia de precios muy altos indican una distribución claramente asimétrica hacia la derecha.

### 4.2 Distribución del precio y transformación logarítmica

Se representan el precio original y su logaritmo. El logaritmo reduce la asimetría y será la transformación empleada como variable objetivo en la regresión.

In [ ]:
# =====================================================
# DISTRIBUCIÓN DEL PRECIO
# =====================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(
    data=df_eda,
    x="price",
    color="#4C72B0",
    bins=60,
    kde=True,
    ax=axes[0],
)
axes[0].set_title("Distribución del precio de oferta")
axes[0].set_xlabel("Precio (€)")
axes[0].set_ylabel("Número de pisos")

df_eda["log_price"] = np.log(df_eda["price"])

sns.histplot(
    data=df_eda,
    x="log_price",
    color="#4C72B0",
    bins=60,
    kde=True,
    ax=axes[1],
)
axes[1].set_title("Distribución del logaritmo del precio")
axes[1].set_xlabel("log(Precio)")
axes[1].set_ylabel("Número de pisos")

plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "figura_3_02_distribucion_precio_logprice.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

print(f"Asimetría del precio: {df_eda['price'].skew():.3f}")
print(f"Asimetría de log(precio): {df_eda['log_price'].skew():.3f}")

### Interpretación

La distribución del precio original presenta una asimetría elevada (**≈ 5,25**). Tras aplicar el logaritmo, la asimetría se reduce de forma notable (**≈ 0,51**).

Por este motivo, `log_price` resulta más adecuada que el precio bruto para la regresión lineal: disminuye el peso de los anuncios más caros y genera una distribución más equilibrada.

## 5. Relación entre variables numéricas y precio

Desde este punto se utiliza la **muestra comparativa de 42.441 pisos**. El filtrado de valores extremos facilita la lectura de los gráficos y mantiene coherencia con la muestra del modelo.

### 5.1 Superficie y precio

Se analiza la relación entre la superficie construida y el precio de oferta.

In [ ]:
# =====================================================
# SUPERFICIE Y PRECIO
# =====================================================

plt.figure(figsize=(10, 6))

sns.regplot(
    data=df_analisis,
    x="size",
    color="#4C72B0",
    y="price",
    scatter_kws={"alpha": 0.20, "s": 15},
    line_kws={"linewidth": 2},
)

plt.title("Relación entre superficie y precio de oferta")
plt.xlabel("Superficie (m²)")
plt.ylabel("Precio (€)")
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "figura_3_04_superficie_precio.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

corr_size = df_analisis["size"].corr(df_analisis["price"])
print(f"Correlación de Pearson: {corr_size:.3f}")

### Interpretación

La superficie mantiene una asociación positiva intensa con el precio. La correlación de Pearson es aproximadamente **0,75**, lo que indica que los pisos de mayor tamaño tienden a registrar precios superiores.

Esta relación es descriptiva y no implica que la superficie sea el único factor relevante; la regresión múltiple permitirá controlar simultáneamente otras características.

### 5.2 Número de habitaciones y baños

Se compara la distribución del precio según el número de habitaciones y baños.

In [ ]:
# =====================================================
# HABITACIONES Y BAÑOS
# =====================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_rooms = df_analisis[df_analisis["rooms"].between(0, 6)]
sns.boxplot(
    data=df_rooms,
    x="rooms",
    color="#4C72B0",
    y="price",
    ax=axes[0],
)
axes[0].set_title("Precio según número de habitaciones")
axes[0].set_xlabel("Habitaciones")
axes[0].set_ylabel("Precio (€)")

df_bath = df_analisis[df_analisis["bathrooms"].between(1, 6)]
sns.boxplot(
    data=df_bath,
    x="bathrooms",
    color="#4C72B0",
    y="price",
    ax=axes[1],
)
axes[1].set_title("Precio según número de baños")
axes[1].set_xlabel("Baños")
axes[1].set_ylabel("Precio (€)")

plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "figura_3_05_habitaciones_banos_precio.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

print(
    f"Correlación rooms-precio: "
    f"{df_analisis['rooms'].corr(df_analisis['price']):.3f}"
)
print(
    f"Correlación bathrooms-precio: "
    f"{df_analisis['bathrooms'].corr(df_analisis['price']):.3f}"
)

### Interpretación

El número de baños presenta una asociación con el precio más intensa que el número de habitaciones. En la muestra comparativa, la correlación con el precio es aproximadamente **0,69 para baños** y **0,30 para habitaciones**.

Parte de estas asociaciones puede estar relacionada con la superficie: las viviendas grandes suelen contar con más habitaciones y baños. Esta posible interrelación se revisará mediante VIF en la modelización.

## 6. Características categóricas y precio

Para las variables con valores `True`, `False` y `NaN`, los gráficos usan únicamente observaciones en las que la característica está informada. De esta forma, un dato ausente no se interpreta de manera automática como ausencia de la característica.

### 6.1 Ascensor, terraza, piscina y jardín

In [ ]:
# =====================================================
# CARACTERÍSTICAS CATEGÓRICAS PRINCIPALES
# =====================================================

variables_grafico = {
    "hasLift": "Ascensor",
    "features/hasTerrace": "Terraza",
    "features/hasSwimmingPool": "Piscina",
    "features/hasGarden": "Jardín",
}

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
axes = axes.ravel()

for ax, (variable, etiqueta) in zip(axes, variables_grafico.items()):
    datos = df_analisis[df_analisis[variable].notna()].copy()

    sns.boxplot(
        data=datos,
        x=variable,
        color="#4C72B0",
        y="price",
        ax=ax,
    )

    ax.set_title(f"Precio según disponibilidad de {etiqueta.lower()}")
    ax.set_xlabel(etiqueta)
    ax.set_ylabel("Precio (€)")
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["No", "Sí"])

plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "figura_3_06_equipamientos_precio.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

resumen_mediana = []

for variable, etiqueta in variables_grafico.items():
    datos = df_analisis[df_analisis[variable].notna()]
    medianas = datos.groupby(variable)["price"].median()

    resumen_mediana.append({
        "Característica": etiqueta,
        "Mediana sin característica": medianas.get(False, np.nan),
        "Mediana con característica": medianas.get(True, np.nan),
    })

tabla_mediana_equipamientos = pd.DataFrame(resumen_mediana).round(0)
display(tabla_mediana_equipamientos)
# Figura 5-1: precio mediano según disponibilidad informada
plot_mediana_equipamientos = (
    pd.DataFrame(resumen_mediana)
    .set_index("Característica")[
        ["Mediana sin característica", "Mediana con característica"]
    ]
    .rename(
        columns={
            "Mediana sin característica": "No",
            "Mediana con característica": "Sí",
        }
    )
)

ax = plot_mediana_equipamientos.plot(
    kind="bar",
    figsize=(11, 6),
)
ax.set_title("Precio mediano de oferta según disponibilidad informada")
ax.set_xlabel("Equipamiento")
ax.set_ylabel("Precio mediano (€)")
plt.xticks(rotation=0)
plt.tight_layout()
ax.figure.savefig(
    FIGURES_DIR / "figura_5_01_mediana_equipamientos.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


### Interpretación

En términos descriptivos, los pisos con ascensor, terraza, piscina o jardín presentan medianas de precio superiores a las de los pisos sin esas características.

Estas diferencias no deben interpretarse como efectos aislados: por ejemplo, una vivienda con piscina también puede ser más grande o encontrarse en una zona más cara. El modelo múltiple permitirá analizar estas características de forma conjunta y estimar su asociación con el precio manteniendo constantes las demás variables incluidas.

### 6.2 Plaza de aparcamiento

En el dataset, `parkingSpace/hasParkingSpace` aparece informado cuando existe el bloque de aparcamiento y sus valores no nulos son afirmativos. Para esta representación se crea una variable auxiliar que distingue entre los anuncios con plaza de aparcamiento informada y aquellos en los que esta característica no aparece especificada.

In [ ]:
# =====================================================
# APARCAMIENTO Y PRECIO
# =====================================================

df_parking = df_analisis.copy()

df_parking["hasParking_eda"] = (
    df_parking["parkingSpace/hasParkingSpace"]
    .eq(True)
    .fillna(False)
)

plt.figure(figsize=(8, 6))

sns.boxplot(
    data=df_parking,
    x="hasParking_eda",
    color="#4C72B0",
    y="price",
)

plt.title("Precio según disponibilidad de plaza de aparcamiento")
plt.xlabel("Plaza de aparcamiento")
plt.ylabel("Precio (€)")
plt.xticks([0, 1], ["No informada", "Sí"])
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "figura_3_07_aparcamiento_precio.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

display(
    df_parking
    .groupby("hasParking_eda")["price"]
    .median()
    .rename("Precio mediano")
    .to_frame()
)

### Interpretación

Los anuncios con plaza de aparcamiento informada presentan una mediana de precio superior. Esta variable debe interpretarse con cautela porque el valor vacío procede de la estructura del anuncio y no de un `False` explícito.

En el modelo se utilizará una codificación derivada específica sin modificar la columna original del dataset.

### 6.3 Estado, aire acondicionado y trastero

Estas variables se incorporarán también a la especificación ampliada del modelo, por lo que se realiza una revisión descriptiva previa.

In [ ]:
# =====================================================
# VARIABLES COMPLEMENTARIAS
# =====================================================

print("Precio mediano según estado:")
display(
    df_analisis
    .groupby("status", dropna=False)["price"]
    .median()
    .sort_values(ascending=False)
    .rename("Precio mediano")
    .to_frame()
)

variables_extra = {
    "features/hasAirConditioning": "Aire acondicionado",
    "features/hasBoxRoom": "Trastero",
}

resumen_extra = []

for variable, etiqueta in variables_extra.items():
    datos = df_analisis[df_analisis[variable].notna()]
    medianas = datos.groupby(variable)["price"].median()

    resumen_extra.append({
        "Característica": etiqueta,
        "Mediana No": medianas.get(False, np.nan),
        "Mediana Sí": medianas.get(True, np.nan),
        "N informados": len(datos),
    })

pd.DataFrame(resumen_extra).round(0)

### Interpretación

Los pisos clasificados como `renew` presentan una mediana de precio inferior a la de los inmuebles en buen estado u obra nueva. El aire acondicionado y el trastero aparecen asociados, asimismo, a precios medianos superiores.

Estas asociaciones ofrecen una justificación descriptiva para evaluar estas variables en el modelo, pero su importancia definitiva dependerá del análisis conjunto con superficie, localización y el resto de características.

## 7. Comparación entre ciudades

### 7.1 Precio mediano

La mediana se utiliza porque resulta menos sensible que la media a los precios extremos.

In [ ]:
# =====================================================
# PRECIO MEDIANO POR CIUDAD
# =====================================================

precio_mediano = (
    df_analisis
    .groupby("ciudad")["price"]
    .median()
    .sort_values(ascending=False)
)

plt.figure(figsize=(10, 6))
sns.barplot(
    x=precio_mediano.values,
    color="#4C72B0",
    y=precio_mediano.index,
)

plt.title("Precio mediano de los pisos por ciudad")
plt.xlabel("Precio mediano (€)")
plt.ylabel("Ciudad")
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "figura_3_08_precio_mediano_ciudad.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

precio_mediano.rename("Precio mediano").to_frame()

### Interpretación

Madrid y Barcelona presentan los mayores precios medianos de la muestra, seguidas por Málaga. En el extremo inferior se sitúan Zaragoza y Ávila.

Estas diferencias ponen de manifiesto que los mercados analizados no presentan un único nivel de precios y respaldan la decisión de estimar regresiones **independientes por ciudad**.

### 7.2 Precio por metro cuadrado

El precio por metro cuadrado posibilita una comparación descriptiva del nivel relativo de precios entre ciudades, ya que expresa el precio de oferta en relación con la superficie de la vivienda.

In [ ]:
# =====================================================
# PRECIO POR METRO CUADRADO
# =====================================================

df_analisis["precio_m2"] = (
    df_analisis["price"] / df_analisis["size"]
)

precio_m2_ciudad = (
    df_analisis
    .groupby("ciudad")["precio_m2"]
    .median()
    .sort_values(ascending=False)
)

plt.figure(figsize=(10, 6))
sns.barplot(
    x=precio_m2_ciudad.values,
    color="#4C72B0",
    y=precio_m2_ciudad.index,
)

plt.title("Precio mediano por metro cuadrado según ciudad")
plt.xlabel("Precio mediano (€/m²)")
plt.ylabel("Ciudad")
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "figura_3_09_precio_m2_ciudad.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

precio_m2_ciudad.rename("Precio mediano €/m²").round(0).to_frame()

### Interpretación

La ordenación se asemeja a la observada con el precio total: Madrid y Barcelona conservan los valores medianos más altos por metro cuadrado, mientras que Ávila registra el nivel más bajo.

`precio_m2` se emplea exclusivamente como indicador descriptivo. **No se incorporará como predictor de `price` o `log_price`**, porque se calcula a partir del propio precio y generaría una relación mecánica con la variable objetivo.

## 8. Correlaciones entre variables numéricas

La matriz recoge únicamente las variables originales usadas como posibles predictores. `precio_m2` queda excluido porque su propia definición incorpora el precio.

In [ ]:
# =====================================================
# MATRIZ DE CORRELACIONES
# =====================================================

variables_corr = [
    "price",
    "size",
    "rooms",
    "bathrooms",
]

corr = df_analisis[variables_corr].corr()

display(corr.round(3))

plt.figure(figsize=(7, 5))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    vmin=-1,
    vmax=1,
    cmap="Blues",
    linewidths=0.5,
    square=True,
)

plt.title("Matriz de correlaciones")
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "figura_3_10_matriz_correlaciones.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

### Interpretación

La superficie presenta la correlación más alta con el precio (**≈ 0,75**), seguida por los baños (**≈ 0,69**). Habitaciones muestra una correlación más moderada (**≈ 0,30**).

Entre algunos predictores también se observa relación: superficie y baños, por ejemplo, están correlacionados. Esto no impide emplearlos conjuntamente, pero justifica verificar la **multicolinealidad mediante VIF** en el Notebook 04.

## 9. Resultados principales

- El dataset procesado contiene **54.004 anuncios**, de los cuales **43.891 son pisos**.
- La muestra comparativa, definida tras exigir valores completos en las variables clave y aplicar el tratamiento de valores extremos, contiene **42.441 pisos**.
- El precio presenta una fuerte asimetría; el logaritmo reduce claramente este problema.
- La superficie es la variable numérica con mayor asociación bivariada con el precio.
- Los baños muestran una relación con el precio más intensa que las habitaciones.
- Ascensor, terraza, piscina, jardín, aparcamiento, aire acondicionado y trastero muestran diferencias descriptivas de precio entre categorías.
- Existen diferencias marcadas de nivel de precios entre ciudades.
- `precio_m2` se mantiene como indicador descriptivo, pero no se utiliza como predictor por estar construido a partir del precio.

## 10. Conclusiones de la fase

El análisis exploratorio indica que el precio de los pisos se vincula con varias dimensiones: superficie, distribución interior, equipamientos, estado del inmueble y mercado urbano.

La superficie presenta la asociación bivariada más alta con el precio, mientras que los baños muestran una relación más intensa que el número de habitaciones. También se aprecian diferencias de precio según la disponibilidad de determinados equipamientos y diferencias claras entre ciudades tanto en precio total como en precio por metro cuadrado.

La transformación logarítmica del precio reduce de forma notable su asimetría y ofrece una variable objetivo más adecuada para la regresión.

Las relaciones observadas son descriptivas y **no deben interpretarse como causales**. Dado que las características de una vivienda se relacionan entre sí, el Notebook 04 utilizará regresiones múltiples independientes por ciudad para estudiar la asociación de cada variable manteniendo constantes las demás variables incluidas en el modelo.

In [ ]:
# =====================================================
# EXPORTACIÓN DE TABLAS DEL ANÁLISIS EXPLORATORIO
# =====================================================

TABLES_DIR = PROJECT_ROOT / "reports" / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

resumen_muestra.round(2).to_csv(
    TABLES_DIR / "03_muestra_por_ciudad.csv",
    index=False,
    encoding="utf-8-sig",
)

estadisticas_precio.to_csv(
    TABLES_DIR / "03_estadisticas_precio.csv",
    encoding="utf-8-sig",
)

tabla_mediana_equipamientos.to_csv(
    TABLES_DIR / "03_mediana_equipamientos.csv",
    index=False,
    encoding="utf-8-sig",
)

tabla_parking = (
    df_parking
    .groupby("hasParking_eda")["price"]
    .median()
    .rename("Precio mediano")
    .reset_index()
)
tabla_parking.to_csv(
    TABLES_DIR / "03_mediana_aparcamiento.csv",
    index=False,
    encoding="utf-8-sig",
)

tabla_estado = (
    df_analisis
    .groupby("status", dropna=False)["price"]
    .median()
    .sort_values(ascending=False)
    .rename("Precio mediano")
    .reset_index()
)
tabla_estado.to_csv(
    TABLES_DIR / "03_mediana_precio_por_estado.csv",
    index=False,
    encoding="utf-8-sig",
)

pd.DataFrame(resumen_extra).round(0).to_csv(
    TABLES_DIR / "03_mediana_variables_complementarias.csv",
    index=False,
    encoding="utf-8-sig",
)

precio_mediano.rename("Precio mediano").reset_index().to_csv(
    TABLES_DIR / "03_precio_mediano_por_ciudad.csv",
    index=False,
    encoding="utf-8-sig",
)

precio_m2_ciudad.rename("Precio mediano €/m²").round(0).reset_index().to_csv(
    TABLES_DIR / "03_precio_m2_mediano_por_ciudad.csv",
    index=False,
    encoding="utf-8-sig",
)

corr.round(3).to_csv(
    TABLES_DIR / "03_matriz_correlaciones.csv",
    encoding="utf-8-sig",
)

print("Tablas del Notebook 03 exportadas en:", TABLES_DIR)
